In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.utils import resample
from sklearn.metrics import root_mean_squared_error
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import Lasso, Ridge
from sklearn.kernel_ridge import KernelRidge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel as C, RBF, RationalQuadratic
import time
import torch
from torch.utils.data import DataLoader, TensorDataset
from src.models.mlp import MLP, create_mlp_pytorch, EarlyStopping
from src.models.resnet import ResBlock, ResNet
import torch.optim as optim

/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
No normalization for SPS. Feature removed!
No normalization for AvgIpc. Feature removed!
No normalization for NumAmideBonds. Feature removed!
No normalization for NumAtomStereoCenters. Feature removed!
No normalization for NumBridgeheadAtoms. Feature removed!
No normalization for NumHeterocycles. Feature removed!
No normalization for NumSpiroAtoms. Feature removed!
No no

In [2]:
descriptor_df = pd.read_csv('data/freesolv/external_descriptors.csv')
smiles_df = pd.read_csv('data/freesolv/smiles.csv')
learned_descriptors_df = pd.read_csv('data/freesolv/learned_predictors_0.csv')
df = pd.concat([learned_descriptors_df, descriptor_df, smiles_df], axis=1)

In [3]:
df.head()

,fp_0,fp_1,fp_2,fp_3,fp_4,fp_5,fp_6,fp_7,fp_8,fp_9,...,209,210,211,212,213,214,215,y,w,ids
0,0.005919,0.0,0.000000,0.0,0.0,0.0,0.0,0.009849,0.030713,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.107729,1.0,C(CCl)OCCCl
1,0.016760,0.0,0.004031,0.0,0.0,0.0,0.0,0.002265,0.022694,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.014446,1.0,C(Cl)(Cl)(Cl)Cl
2,0.006240,0.0,0.010199,0.0,0.0,0.0,0.0,0.007726,0.026535,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.199502,1.0,CC(C)CC(=O)C
3,0.007206,0.0,0.005383,0.0,0.0,0.0,0.0,0.005392,0.038531,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.449453,1.0,CCCCO[N+](=O)[O-]
4,0.003805,0.0,0.006889,0.0,0.0,0.0,0.0,0.004972,0.007662,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.574428,1.0,CSC


In [4]:
df = df.drop(columns='ids')
df.head()

,fp_0,fp_1,fp_2,fp_3,fp_4,fp_5,fp_6,fp_7,fp_8,fp_9,...,208,209,210,211,212,213,214,215,y,w
0,0.005919,0.0,0.000000,0.0,0.0,0.0,0.0,0.009849,0.030713,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.107729,1.0
1,0.016760,0.0,0.004031,0.0,0.0,0.0,0.0,0.002265,0.022694,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.014446,1.0
2,0.006240,0.0,0.010199,0.0,0.0,0.0,0.0,0.007726,0.026535,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.199502,1.0
3,0.007206,0.0,0.005383,0.0,0.0,0.0,0.0,0.005392,0.038531,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.449453,1.0
4,0.003805,0.0,0.006889,0.0,0.0,0.0,0.0,0.004972,0.007662,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.574428,1.0


In [5]:
df['w'].unique()

array([1.])

In [6]:
df = df.drop(columns='w')
df.head()

,fp_0,fp_1,fp_2,fp_3,fp_4,fp_5,fp_6,fp_7,fp_8,fp_9,...,207,208,209,210,211,212,213,214,215,y
0,0.005919,0.0,0.000000,0.0,0.0,0.0,0.0,0.009849,0.030713,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.107729
1,0.016760,0.0,0.004031,0.0,0.0,0.0,0.0,0.002265,0.022694,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.014446
2,0.006240,0.0,0.010199,0.0,0.0,0.0,0.0,0.007726,0.026535,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.199502
3,0.007206,0.0,0.005383,0.0,0.0,0.0,0.0,0.005392,0.038531,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.449453
4,0.003805,0.0,0.006889,0.0,0.0,0.0,0.0,0.004972,0.007662,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.574428


In [7]:
X = df.drop(columns=['y'])
y = df['y']

In [8]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5)

In [9]:
models_df = pd.DataFrame(columns=[
    'index', 'model_type', 'hyperparams', 'rmse'
])
models_list = []
config_id = 0
base_seed = 42

In [10]:
for i in range(4):
    seed = base_seed + config_id
    X_boot, y_boot = resample(
    X_train, y_train,
    replace=True,
    n_samples=len(X_train),
    random_state=seed)
    scaler = StandardScaler()
    X_boot_scaled = scaler.fit_transform(X_boot)
    model = KNeighborsRegressor()
    model.fit(X_boot_scaled, y_boot)
    X_val_scaled = scaler.transform(X_val)
    y_val_pred = model.predict(X_val_scaled)
    rmse = root_mean_squared_error(y_val, y_val_pred)

    model_metadata = {
        'index': config_id,
        'model_type': type(model).__name__,
        'hyperparams': {},
        'rmse': rmse
    }
    config_id += 1
    
    print(model_metadata)
    models_list.append(model_metadata)
 

{'index': 0, 'model_type': 'KNeighborsRegressor', 'hyperparams': {}, 'rmse': 0.30109799755193034}
{'index': 1, 'model_type': 'KNeighborsRegressor', 'hyperparams': {}, 'rmse': 0.4657266303774585}
{'index': 2, 'model_type': 'KNeighborsRegressor', 'hyperparams': {}, 'rmse': 0.4090489627021739}
{'index': 3, 'model_type': 'KNeighborsRegressor', 'hyperparams': {}, 'rmse': 0.33176904431885146}


In [11]:
param_grid = {
    'alpha': [0.001, 0.002, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2]
}

for params in ParameterGrid(param_grid):
    seed = base_seed + config_id
    X_boot, y_boot = resample(
    X_train, y_train,
    replace=True,
    n_samples=len(X_train),
    random_state=seed)
    scaler = StandardScaler()
    X_boot_scaled = scaler.fit_transform(X_boot)
    model = Lasso(**params)
    model.fit(X_boot_scaled, y_boot)
    X_val_scaled = scaler.transform(X_val)
    y_val_pred = model.predict(X_val_scaled)
    rmse = root_mean_squared_error(y_val, y_val_pred)

    model_metadata = {
        'index': config_id,
        'model_type': type(model).__name__,
        'hyperparams': params,
        'rmse': rmse
    }
    config_id += 1
    
    print(model_metadata)
    models_list.append(model_metadata)

{'index': 4, 'model_type': 'Lasso', 'hyperparams': {'alpha': 0.001}, 'rmse': 0.26280552998679335}
{'index': 5, 'model_type': 'Lasso', 'hyperparams': {'alpha': 0.002}, 'rmse': 0.25729733201674054}
{'index': 6, 'model_type': 'Lasso', 'hyperparams': {'alpha': 0.005}, 'rmse': 0.29671325918853586}
{'index': 7, 'model_type': 'Lasso', 'hyperparams': {'alpha': 0.01}, 'rmse': 0.26053927565601315}
{'index': 8, 'model_type': 'Lasso', 'hyperparams': {'alpha': 0.02}, 'rmse': 0.28471073997471036}
{'index': 9, 'model_type': 'Lasso', 'hyperparams': {'alpha': 0.05}, 'rmse': 0.34324741130191005}
{'index': 10, 'model_type': 'Lasso', 'hyperparams': {'alpha': 0.1}, 'rmse': 0.3774793261411331}
{'index': 11, 'model_type': 'Lasso', 'hyperparams': {'alpha': 0.2}, 'rmse': 0.5313726916801317}


/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.903e-01, tolerance: 5.046e-02
  model = cd_fast.enet_coordinate_descent(
/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.095e-01, tolerance: 5.149e-02
  model = cd_fast.enet_coordinate_descent(


In [12]:
param_grid = {
    'alpha': [1, 2, 3, 5, 7, 10, 15, 20, 30, 50]
}

for params in ParameterGrid(param_grid):
    seed = base_seed + config_id
    X_boot, y_boot = resample(
    X_train, y_train,
    replace=True,
    n_samples=len(X_train),
    random_state=seed)
    scaler = StandardScaler()
    X_boot_scaled = scaler.fit_transform(X_boot)
    model = Ridge(**params)
    model.fit(X_boot_scaled, y_boot)
    X_val_scaled = scaler.transform(X_val)
    y_val_pred = model.predict(X_val_scaled)
    rmse = root_mean_squared_error(y_val, y_val_pred)

    model_metadata = {
        'index': config_id,
        'model_type': type(model).__name__,
        'hyperparams': params,
        'rmse': rmse
    }
    config_id += 1
    
    print(model_metadata)
    models_list.append(model_metadata)

{'index': 12, 'model_type': 'Ridge', 'hyperparams': {'alpha': 1}, 'rmse': 0.274791406275937}
{'index': 13, 'model_type': 'Ridge', 'hyperparams': {'alpha': 2}, 'rmse': 0.2667811078874242}
{'index': 14, 'model_type': 'Ridge', 'hyperparams': {'alpha': 3}, 'rmse': 0.32722851294618766}
{'index': 15, 'model_type': 'Ridge', 'hyperparams': {'alpha': 5}, 'rmse': 0.27431612464288047}
{'index': 16, 'model_type': 'Ridge', 'hyperparams': {'alpha': 7}, 'rmse': 0.2583906647995346}
{'index': 17, 'model_type': 'Ridge', 'hyperparams': {'alpha': 10}, 'rmse': 0.2653915182964347}
{'index': 18, 'model_type': 'Ridge', 'hyperparams': {'alpha': 15}, 'rmse': 0.3008588245844687}
{'index': 19, 'model_type': 'Ridge', 'hyperparams': {'alpha': 20}, 'rmse': 0.2622551851519705}
{'index': 20, 'model_type': 'Ridge', 'hyperparams': {'alpha': 30}, 'rmse': 0.8964576365455814}
{'index': 21, 'model_type': 'Ridge', 'hyperparams': {'alpha': 50}, 'rmse': 0.7918415432135305}


In [13]:
param_grid = [
    # Polynomial kernel (3rd degree)
    {
        'kernel': ['poly'],
        'alpha': [0.1, 1, 10],
        'degree': [3],
    },
    # RBF kernel
    {
        'kernel': ['rbf'],
        'alpha': [0.1, 1, 10],
    }
]

for idx, params in enumerate(ParameterGrid(param_grid)):
    seed = base_seed + config_id
    X_boot, y_boot = resample(
    X_train, y_train,
    replace=True,
    n_samples=len(X_train),
    random_state=seed)
    scaler = StandardScaler()
    X_boot_scaled = scaler.fit_transform(X_boot)
    model = KernelRidge(**params)
    model.fit(X_boot_scaled, y_boot)
    X_val_scaled = scaler.transform(X_val)
    y_val_pred = model.predict(X_val_scaled)
    rmse = root_mean_squared_error(y_val, y_val_pred)

    model_metadata = {
        'index': config_id,
        'model_type': type(model).__name__,
        'hyperparams': params,
        'rmse': rmse
    }
    config_id += 1
    
    print(model_metadata)
    models_list.append(model_metadata)

{'index': 22, 'model_type': 'KernelRidge', 'hyperparams': {'alpha': 0.1, 'degree': 3, 'kernel': 'poly'}, 'rmse': 0.30413401701908416}
{'index': 23, 'model_type': 'KernelRidge', 'hyperparams': {'alpha': 1, 'degree': 3, 'kernel': 'poly'}, 'rmse': 0.38796143986421605}
{'index': 24, 'model_type': 'KernelRidge', 'hyperparams': {'alpha': 10, 'degree': 3, 'kernel': 'poly'}, 'rmse': 0.8359538667074695}
{'index': 25, 'model_type': 'KernelRidge', 'hyperparams': {'alpha': 0.1, 'kernel': 'rbf'}, 'rmse': 0.35591709148388817}
{'index': 26, 'model_type': 'KernelRidge', 'hyperparams': {'alpha': 1, 'kernel': 'rbf'}, 'rmse': 0.4369372938666854}
{'index': 27, 'model_type': 'KernelRidge', 'hyperparams': {'alpha': 10, 'kernel': 'rbf'}, 'rmse': 0.6477949048304251}


In [14]:
param_grid = [
    {'max_depth': [3], 'n_estimators': [10]},
    {'max_depth': [5], 'n_estimators': [10]},
    {'max_depth': [5], 'n_estimators': [20]},
    {'max_depth': [5], 'n_estimators': [50]},
    {'max_depth': [8], 'n_estimators': [50]},
    {'max_depth': [8], 'n_estimators': [100]},
    {'max_depth': [5], 'n_estimators': [100]},
    {'max_depth': [8], 'n_estimators': [100]},
    {'max_depth': [16], 'n_estimators': [200]},
    {'max_depth': [32], 'n_estimators': [1000]}
]
for idx, params in enumerate(ParameterGrid(param_grid)):
    seed = base_seed + config_id
    X_boot, y_boot = resample(
    X_train, y_train,
    replace=True,
    n_samples=len(X_train),
    random_state=seed)
    scaler = StandardScaler()
    X_boot_scaled = scaler.fit_transform(X_boot)
    model = RandomForestRegressor(**params)
    model.fit(X_boot_scaled, y_boot)
    X_val_scaled = scaler.transform(X_val)
    y_val_pred = model.predict(X_val_scaled)
    rmse = root_mean_squared_error(y_val, y_val_pred)

    model_metadata = {
        'index': config_id,
        'model_type': type(model).__name__,
        'hyperparams': params,
        'rmse': rmse
    }
    config_id += 1
    
    print(model_metadata)
    models_list.append(model_metadata)

{'index': 28, 'model_type': 'RandomForestRegressor', 'hyperparams': {'max_depth': 3, 'n_estimators': 10}, 'rmse': 0.48316045829267057}
{'index': 29, 'model_type': 'RandomForestRegressor', 'hyperparams': {'max_depth': 5, 'n_estimators': 10}, 'rmse': 0.36481943858227034}
{'index': 30, 'model_type': 'RandomForestRegressor', 'hyperparams': {'max_depth': 5, 'n_estimators': 20}, 'rmse': 0.2897772097917187}
{'index': 31, 'model_type': 'RandomForestRegressor', 'hyperparams': {'max_depth': 5, 'n_estimators': 50}, 'rmse': 0.31424632353868553}
{'index': 32, 'model_type': 'RandomForestRegressor', 'hyperparams': {'max_depth': 8, 'n_estimators': 50}, 'rmse': 0.26891537521403913}
{'index': 33, 'model_type': 'RandomForestRegressor', 'hyperparams': {'max_depth': 8, 'n_estimators': 100}, 'rmse': 0.3050162246606871}
{'index': 34, 'model_type': 'RandomForestRegressor', 'hyperparams': {'max_depth': 5, 'n_estimators': 100}, 'rmse': 0.45434734706834556}
{'index': 35, 'model_type': 'RandomForestRegressor', 'h

In [15]:
hyperparams_list = [
    {'max_depth': 3, 'n_estimators': 100, 'learning_rate': 0.1},
    {'max_depth': 5, 'n_estimators': 100, 'learning_rate': 0.1},
    {'max_depth': 3, 'n_estimators': 200, 'learning_rate': 0.1},
    {'max_depth': 10, 'n_estimators': 200, 'learning_rate': 0.2},
    {'max_depth': 5, 'n_estimators': 500, 'learning_rate': 0.1},
    {'max_depth': 10, 'n_estimators': 500, 'learning_rate': 0.01},
    {'max_depth': 10, 'n_estimators': 1000, 'learning_rate': 0.1},
    {'max_depth': 32, 'n_estimators': 2000, 'learning_rate': 0.1}
]

for idx, params in enumerate(hyperparams_list):

    seed = base_seed + config_id

    X_boot, y_boot = resample(
    X_train, y_train,
    replace=True,
    n_samples=len(X_train),
    random_state=seed)

    scaler = StandardScaler()
    X_boot_scaled = scaler.fit_transform(X_boot)

    start_time = time.time()
    model = XGBRegressor(
        max_depth=params['max_depth'],
        n_estimators=params['n_estimators'],
        learning_rate=params['learning_rate'],
        n_jobs=-1,
    )

    model.fit(X_boot_scaled, y_boot)

    X_val_scaled = scaler.transform(X_val)
    y_val_pred = model.predict(X_val_scaled)
    
    rmse = root_mean_squared_error(y_val, y_val_pred)
    training_time = time.time() - start_time
    
    print(f'Training time: {training_time}')
    
    model_metadata = {
        'index': config_id,
        'model_type': type(model).__name__,
        'hyperparams': params,
        'rmse': rmse
    }
    
    config_id += 1
    
    print(model_metadata)
    
    models_list.append(model_metadata)

Training time: 0.17751383781433105
{'index': 38, 'model_type': 'XGBRegressor', 'hyperparams': {'max_depth': 3, 'n_estimators': 100, 'learning_rate': 0.1}, 'rmse': 0.3173875292248438}
Training time: 0.2584071159362793
{'index': 39, 'model_type': 'XGBRegressor', 'hyperparams': {'max_depth': 5, 'n_estimators': 100, 'learning_rate': 0.1}, 'rmse': 0.232863508655864}
Training time: 0.22453999519348145
{'index': 40, 'model_type': 'XGBRegressor', 'hyperparams': {'max_depth': 3, 'n_estimators': 200, 'learning_rate': 0.1}, 'rmse': 0.30695122712655953}
Training time: 0.5219578742980957
{'index': 41, 'model_type': 'XGBRegressor', 'hyperparams': {'max_depth': 10, 'n_estimators': 200, 'learning_rate': 0.2}, 'rmse': 0.3395619987867347}
Training time: 0.8248302936553955
{'index': 42, 'model_type': 'XGBRegressor', 'hyperparams': {'max_depth': 5, 'n_estimators': 500, 'learning_rate': 0.1}, 'rmse': 0.3636496691367162}
Training time: 5.013906002044678
{'index': 43, 'model_type': 'XGBRegressor', 'hyperpara

In [16]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel as C, RBF, RationalQuadratic

param_grid = [
    # Matern kernel models
    {
        'kernel': Matern(),  
    },
    {
        'kernel': Matern(),  
    },
    # Quadratic (RBF) kernel models  
    {
        'kernel': RBF(),  
    },
    {
        'kernel': RBF(),
    }
]

for idx, params in enumerate(param_grid):
    seed = base_seed + config_id
    X_boot, y_boot = resample(
    X_train, y_train,
    replace=True,
    n_samples=len(X_train),
    random_state=seed)
    scaler = StandardScaler()
    X_boot_scaled = scaler.fit_transform(X_boot)
    start_time = time.time()
    model = GaussianProcessRegressor(kernel=params['kernel'])
    model.fit(X_boot_scaled, y_boot)
    X_val_scaled = scaler.transform(X_val)
    y_val_pred = model.predict(X_val_scaled)
    rmse = root_mean_squared_error(y_val, y_val_pred)
    training_time = time.time() - start_time
    print(f'Training time: {training_time}')
    model_metadata = {
        'index': config_id,
        'model_type': type(model).__name__,
        'hyperparams': params,
        'rmse': rmse
    }
    config_id += 1
    
    print(model_metadata)
    models_list.append(model_metadata)


/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/sklearn/gaussian_process/_gpr.py:663: ConvergenceWarning: lbfgs failed to converge after 6 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)


Training time: 1.8506176471710205
{'index': 46, 'model_type': 'GaussianProcessRegressor', 'hyperparams': {'kernel': Matern(length_scale=1, nu=1.5)}, 'rmse': 0.20052852390943635}
Training time: 0.5480823516845703
{'index': 47, 'model_type': 'GaussianProcessRegressor', 'hyperparams': {'kernel': Matern(length_scale=1, nu=1.5)}, 'rmse': 0.21974053165354152}
Training time: 1.2855401039123535
{'index': 48, 'model_type': 'GaussianProcessRegressor', 'hyperparams': {'kernel': RBF(length_scale=1)}, 'rmse': 0.24904687682427573}
Training time: 0.6075630187988281
{'index': 49, 'model_type': 'GaussianProcessRegressor', 'hyperparams': {'kernel': RBF(length_scale=1)}, 'rmse': 0.2371973841890745}


In [17]:
import torch.nn as nn

In [18]:
hyperparams_list = [
    {'n_layers': 2, 'layer_size': 5, 'lr': 0.01, 'l2_reg': 0.01},
    {'n_layers': 2, 'layer_size': 10, 'lr': 0.001, 'l2_reg': 0.1},
    {'n_layers': 3, 'layer_size': 10, 'lr': 0.001, 'l2_reg': 0.1},
    {'n_layers': 3, 'layer_size': 10, 'lr': 0.0005, 'l2_reg': 0.1}
]
for idx, params in enumerate(hyperparams_list):
    
    seed = base_seed + config_id
    X_boot, y_boot = resample(
    X_train, y_train,
    replace=True,
    n_samples=len(X_train),
    random_state=seed)

    params.update({
    'input_dim': X_boot.shape[1],
    'output_dim': 1
})
    scaler = StandardScaler()
    X_boot_scaled = scaler.fit_transform(X_boot)

    X_boot_tensor = torch.FloatTensor(X_boot_scaled)
    y_boot_tensor = torch.FloatTensor(y_boot.values)
    train_dataset = TensorDataset(X_boot_tensor, y_boot_tensor)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

    X_val_scaled = scaler.transform(X_val)
    X_val_tensor = torch.FloatTensor(X_val_scaled)
    y_val_tensor = torch.FloatTensor(y_val.values)
    val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
    val_loader = DataLoader(val_dataset, batch_size=32)

    # Create MLP
    model, optimizer = create_mlp_pytorch(
        params['input_dim'],
        params['output_dim'], 
        n_layers=params['n_layers'],
        layer_size=params['layer_size'],
        lr=params['lr'],
        l2_reg=params['l2_reg'])
    
    # Training code would go here
    # model.train() ... etc.
    # Basic EarlyStopping for 1000 epochs
    early_stopping = EarlyStopping(patience=20, min_delta=0.001, verbose=True)
    criterion = nn.MSELoss()
  
    train_losses = []
    val_losses = []
    
    for epoch in range(1000):
        # Training phase
        model.train()
        train_loss = 0.0
        for batch_idx, (data, target) in enumerate(train_loader):
            
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        avg_train_loss = train_loss / len(train_loader)
        train_losses.append(avg_train_loss)
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for data, target in val_loader:
                output = model(data)
                val_loss += criterion(output, target).item()
        
        avg_val_loss = val_loss / len(val_loader)
        val_losses.append(avg_val_loss)
        
        print(f'Epoch {epoch+1}/{1000}, '
                f'Train Loss: {avg_train_loss:.4f}, '
                f'Val Loss: {avg_val_loss:.4f}')
        
        # Early stopping check
        if early_stopping(avg_val_loss, model):
            print("Early stopping triggered. Restoring best model...")
            # Restore the best model
            model.load_state_dict(early_stopping.best_model_state)
            break
    
    model.eval()

    all_predictions = []
    all_targets = []

    with torch.no_grad():
        for batch_idx, (data, target) in enumerate(val_loader):
            predictions = model(data)
            
            all_predictions.append(predictions.cpu().numpy())
            all_targets.append(target.numpy())

    # Concatenate all batches
    all_predictions = np.concatenate(all_predictions, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)

    # Calculate RMSE
    rmse = root_mean_squared_error(all_targets, all_predictions)
    
    # Store metadata
    model_metadata = {
        'index': config_id,
        'model_type': 'MLP_PyTorch',
        'hyperparams': params,
        'rmse': rmse
    }
    config_id += 1
    print(model_metadata)
    models_list.append(model_metadata)

Epoch 1/1000, Train Loss: 1.0674, Val Loss: 1.0285
Validation loss decreased (inf -> 1.028459). Saving model...
Epoch 2/1000, Train Loss: 0.9327, Val Loss: 1.0045
Validation loss decreased (1.028459 -> 1.004511). Saving model...
Epoch 3/1000, Train Loss: 0.9481, Val Loss: 1.0012
Validation loss decreased (1.004511 -> 1.001207). Saving model...
Epoch 4/1000, Train Loss: 0.9251, Val Loss: 1.0202
EarlyStopping counter: 1 out of 20


/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([32])) that is different to the input size (torch.Size([32, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([1, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 5/1000, Train Loss: 0.9195, Val Loss: 1.0216
EarlyStopping counter: 2 out of 20
Epoch 6/1000, Train Loss: 1.0213, Val Loss: 1.0089
EarlyStopping counter: 3 out of 20
Epoch 7/1000, Train Loss: 0.9525, Val Loss: 1.0127
EarlyStopping counter: 4 out of 20
Epoch 8/1000, Train Loss: 0.9594, Val Loss: 1.0142
EarlyStopping counter: 5 out of 20
Epoch 9/1000, Train Loss: 0.9233, Val Loss: 1.0129
EarlyStopping counter: 6 out of 20
Epoch 10/1000, Train Loss: 0.9648, Val Loss: 1.0087
EarlyStopping counter: 7 out of 20
Epoch 11/1000, Train Loss: 0.9134, Val Loss: 1.0379
EarlyStopping counter: 8 out of 20
Epoch 12/1000, Train Loss: 0.9164, Val Loss: 1.0059
EarlyStopping counter: 9 out of 20
Epoch 13/1000, Train Loss: 0.9166, Val Loss: 1.0083
EarlyStopping counter: 10 out of 20
Epoch 14/1000, Train Loss: 0.9181, Val Loss: 1.0126
EarlyStopping counter: 11 out of 20
Epoch 15/1000, Train Loss: 1.0435, Val Loss: 1.0155
EarlyStopping counter: 12 out of 20
Epoch 16/1000, Train Loss: 0.9578, Val Loss: 

/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([32])) that is different to the input size (torch.Size([32, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([1, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 13/1000, Train Loss: 0.7681, Val Loss: 1.0196
EarlyStopping counter: 12 out of 20
Epoch 14/1000, Train Loss: 0.8221, Val Loss: 1.0208
EarlyStopping counter: 13 out of 20
Epoch 15/1000, Train Loss: 0.8925, Val Loss: 1.0261
EarlyStopping counter: 14 out of 20
Epoch 16/1000, Train Loss: 0.7735, Val Loss: 1.0207
EarlyStopping counter: 15 out of 20
Epoch 17/1000, Train Loss: 0.7756, Val Loss: 1.0205
EarlyStopping counter: 16 out of 20
Epoch 18/1000, Train Loss: 0.8906, Val Loss: 1.0212
EarlyStopping counter: 17 out of 20
Epoch 19/1000, Train Loss: 0.8421, Val Loss: 1.0187
EarlyStopping counter: 18 out of 20
Epoch 20/1000, Train Loss: 0.7734, Val Loss: 1.0179
EarlyStopping counter: 19 out of 20
Epoch 21/1000, Train Loss: 0.7911, Val Loss: 1.0202
EarlyStopping counter: 20 out of 20
Early stopping triggered
Early stopping triggered. Restoring best model...
{'index': 51, 'model_type': 'MLP_PyTorch', 'hyperparams': {'n_layers': 2, 'layer_size': 10, 'lr': 0.001, 'l2_reg': 0.1, 'input_dim': 

/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([32])) that is different to the input size (torch.Size([32, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([1, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 6/1000, Train Loss: 0.9201, Val Loss: 1.0300
Validation loss decreased (1.031632 -> 1.030023). Saving model...
Epoch 7/1000, Train Loss: 0.9161, Val Loss: 1.0313
EarlyStopping counter: 1 out of 20
Epoch 8/1000, Train Loss: 0.9271, Val Loss: 1.0297
EarlyStopping counter: 2 out of 20
Epoch 9/1000, Train Loss: 0.9756, Val Loss: 1.0295
EarlyStopping counter: 3 out of 20
Epoch 10/1000, Train Loss: 0.9780, Val Loss: 1.0258
Validation loss decreased (1.030023 -> 1.025794). Saving model...
Epoch 11/1000, Train Loss: 1.0348, Val Loss: 1.0245
Validation loss decreased (1.025794 -> 1.024525). Saving model...
Epoch 12/1000, Train Loss: 0.9399, Val Loss: 1.0294
EarlyStopping counter: 1 out of 20
Epoch 13/1000, Train Loss: 0.9183, Val Loss: 1.0263
EarlyStopping counter: 2 out of 20
Epoch 14/1000, Train Loss: 1.0740, Val Loss: 1.0264
EarlyStopping counter: 3 out of 20
Epoch 15/1000, Train Loss: 0.9224, Val Loss: 1.0285
EarlyStopping counter: 4 out of 20
Epoch 16/1000, Train Loss: 0.9152, Val Lo

/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([32])) that is different to the input size (torch.Size([32, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([1, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 6/1000, Train Loss: 0.9866, Val Loss: 1.0606
Validation loss decreased (1.065639 -> 1.060607). Saving model...
Epoch 7/1000, Train Loss: 1.0027, Val Loss: 1.0562
Validation loss decreased (1.060607 -> 1.056243). Saving model...
Epoch 8/1000, Train Loss: 1.0048, Val Loss: 1.0538
Validation loss decreased (1.056243 -> 1.053829). Saving model...
Epoch 9/1000, Train Loss: 0.9863, Val Loss: 1.0488
Validation loss decreased (1.053829 -> 1.048750). Saving model...
Epoch 10/1000, Train Loss: 1.0440, Val Loss: 1.0451
Validation loss decreased (1.048750 -> 1.045103). Saving model...
Epoch 11/1000, Train Loss: 0.9803, Val Loss: 1.0416
Validation loss decreased (1.045103 -> 1.041569). Saving model...
Epoch 12/1000, Train Loss: 0.9823, Val Loss: 1.0396
Validation loss decreased (1.041569 -> 1.039577). Saving model...
Epoch 13/1000, Train Loss: 1.0252, Val Loss: 1.0379
Validation loss decreased (1.039577 -> 1.037902). Saving model...
Epoch 14/1000, Train Loss: 0.9760, Val Loss: 1.0371
EarlySto

In [19]:
def create_mlp_resnet(input_dim, output_dim, block_dim, hidden_dim, num_blocks, lr, l2_reg):
    model = ResNet(
        input_dim = input_dim,
        output_dim = output_dim,
        num_blocks = num_blocks,
        hidden_dim = hidden_dim,
        block_dim = block_dim
    )
    optimizer = optim.Adam(model.parameters(), lr, weight_decay=l2_reg)

    return model, optimizer

In [20]:
specified_configs = [
    # D block D hidden N blocks Learning rate L2 regularisation
    {'block_dim': 16, 'hidden_dim': 8, 'num_blocks': 2, 'lr': 0.01, 'l2_reg': 0.01},
    {'block_dim': 32, 'hidden_dim': 16, 'num_blocks': 2, 'lr': 0.01, 'l2_reg': 0.1},
    {'block_dim': 64, 'hidden_dim': 32, 'num_blocks': 3, 'lr': 0.001, 'l2_reg': 0.01},
    {'block_dim': 64, 'hidden_dim': 32, 'num_blocks': 3, 'lr': 0.001, 'l2_reg': 0.1},
]

for idx, params in enumerate(specified_configs):
    seed = base_seed + config_id
    X_boot, y_boot = resample(
    X_train, y_train,
    replace=True,
    n_samples=len(X_train),
    random_state=seed)
    params.update({
    'input_dim': X_boot.shape[1],
    'output_dim': 1
})
    scaler = StandardScaler()
    X_boot_scaled = scaler.fit_transform(X_boot)
    X_boot_tensor = torch.FloatTensor(X_boot_scaled)
    y_boot_tensor = torch.FloatTensor(y_boot.values)
    train_dataset = TensorDataset(X_boot_tensor, y_boot_tensor)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

    X_val_scaled = scaler.transform(X_val)
    X_val_tensor = torch.FloatTensor(X_val_scaled)
    y_val_tensor = torch.FloatTensor(y_val.values)
    val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
    val_loader = DataLoader(val_dataset, batch_size=32)

    model, optimizer = create_mlp_resnet(
        input_dim = params['input_dim'], 
        output_dim = params['output_dim'],
        num_blocks=params['num_blocks'],
        hidden_dim=params['hidden_dim'],
        block_dim=params['block_dim'],
        lr=params['lr'],
        l2_reg=params['l2_reg'])

    early_stopping = EarlyStopping(patience=20, min_delta=0.001, verbose=True)
    criterion = nn.MSELoss()

    early_stopping = EarlyStopping(patience=20, min_delta=0.001, verbose=True)
    criterion = nn.MSELoss()
  
    train_losses = []
    val_losses = []
    
    for epoch in range(1000):
        # Training phase
        model.train()
        train_loss = 0.0
        for batch_idx, (data, target) in enumerate(train_loader):
            
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        avg_train_loss = train_loss / len(train_loader)
        train_losses.append(avg_train_loss)
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for data, target in val_loader:
                output = model(data)
                val_loss += criterion(output, target).item()
        
        avg_val_loss = val_loss / len(val_loader)
        val_losses.append(avg_val_loss)
        
        print(f'Epoch {epoch+1}/{1000}, '
                f'Train Loss: {avg_train_loss:.4f}, '
                f'Val Loss: {avg_val_loss:.4f}')
        
        # Early stopping check
        if early_stopping(avg_val_loss, model):
            print("Early stopping triggered. Restoring best model...")
            # Restore the best model
            model.load_state_dict(early_stopping.best_model_state)
            break
    
    model.eval()

    all_predictions = []
    all_targets = []

    with torch.no_grad():
        for batch_idx, (data, target) in enumerate(val_loader):
            predictions = model(data)
            
            all_predictions.append(predictions.cpu().numpy())
            all_targets.append(target.numpy())

    # Concatenate all batches
    all_predictions = np.concatenate(all_predictions, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)

    # Calculate RMSE
    rmse = root_mean_squared_error(all_targets, all_predictions)
    
    # Store metadata
    model_metadata = {
        'index': config_id,
        'model_type': 'ResNet',
        'hyperparams': None,
        'rmse': rmse
    }
    config_id += 1
    print(model_metadata)
    models_list.append(model_metadata)

Epoch 1/1000, Train Loss: 1.0931, Val Loss: 1.2276
Validation loss decreased (inf -> 1.227566). Saving model...
Epoch 2/1000, Train Loss: 1.0475, Val Loss: 1.0180
Validation loss decreased (1.227566 -> 1.018016). Saving model...
Epoch 3/1000, Train Loss: 0.9632, Val Loss: 1.0230
EarlyStopping counter: 1 out of 20


/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([32])) that is different to the input size (torch.Size([32, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([1, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 4/1000, Train Loss: 0.9669, Val Loss: 1.1002
EarlyStopping counter: 2 out of 20
Epoch 5/1000, Train Loss: 1.1287, Val Loss: 1.0309
EarlyStopping counter: 3 out of 20
Epoch 6/1000, Train Loss: 1.0124, Val Loss: 1.0620
EarlyStopping counter: 4 out of 20
Epoch 7/1000, Train Loss: 0.9553, Val Loss: 1.0538
EarlyStopping counter: 5 out of 20
Epoch 8/1000, Train Loss: 0.9513, Val Loss: 1.0848
EarlyStopping counter: 6 out of 20
Epoch 9/1000, Train Loss: 0.9841, Val Loss: 1.0194
EarlyStopping counter: 7 out of 20
Epoch 10/1000, Train Loss: 0.9493, Val Loss: 1.0011
Validation loss decreased (1.018016 -> 1.001072). Saving model...
Epoch 11/1000, Train Loss: 0.9846, Val Loss: 1.0223
EarlyStopping counter: 1 out of 20
Epoch 12/1000, Train Loss: 0.9537, Val Loss: 1.0058
EarlyStopping counter: 2 out of 20
Epoch 13/1000, Train Loss: 0.9558, Val Loss: 1.0480
EarlyStopping counter: 3 out of 20
Epoch 14/1000, Train Loss: 0.9506, Val Loss: 1.0261
EarlyStopping counter: 4 out of 20
Epoch 15/1000, Tra

/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([32])) that is different to the input size (torch.Size([32, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([1, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 7/1000, Train Loss: 0.9290, Val Loss: 1.0083
EarlyStopping counter: 5 out of 20
Epoch 8/1000, Train Loss: 0.9100, Val Loss: 1.0346
EarlyStopping counter: 6 out of 20
Epoch 9/1000, Train Loss: 0.9740, Val Loss: 1.0143
EarlyStopping counter: 7 out of 20
Epoch 10/1000, Train Loss: 0.9260, Val Loss: 1.0179
EarlyStopping counter: 8 out of 20
Epoch 11/1000, Train Loss: 1.0282, Val Loss: 1.0142
EarlyStopping counter: 9 out of 20
Epoch 12/1000, Train Loss: 0.9093, Val Loss: 1.0251
EarlyStopping counter: 10 out of 20
Epoch 13/1000, Train Loss: 0.9196, Val Loss: 1.0074
EarlyStopping counter: 11 out of 20
Epoch 14/1000, Train Loss: 0.9806, Val Loss: 1.0508
EarlyStopping counter: 12 out of 20
Epoch 15/1000, Train Loss: 0.9749, Val Loss: 1.0279
EarlyStopping counter: 13 out of 20
Epoch 16/1000, Train Loss: 0.9372, Val Loss: 1.0072
EarlyStopping counter: 14 out of 20
Epoch 17/1000, Train Loss: 0.9424, Val Loss: 1.0121
EarlyStopping counter: 15 out of 20
Epoch 18/1000, Train Loss: 0.9066, Val L

/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([32])) that is different to the input size (torch.Size([32, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([1, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 8/1000, Train Loss: 0.8417, Val Loss: 1.0118
EarlyStopping counter: 6 out of 20
Epoch 9/1000, Train Loss: 0.8368, Val Loss: 1.0103
EarlyStopping counter: 7 out of 20
Epoch 10/1000, Train Loss: 0.8653, Val Loss: 1.0130
EarlyStopping counter: 8 out of 20
Epoch 11/1000, Train Loss: 0.9755, Val Loss: 1.0059
Validation loss decreased (1.009888 -> 1.005888). Saving model...
Epoch 12/1000, Train Loss: 0.9397, Val Loss: 1.0395
EarlyStopping counter: 1 out of 20
Epoch 13/1000, Train Loss: 0.8619, Val Loss: 1.0213
EarlyStopping counter: 2 out of 20
Epoch 14/1000, Train Loss: 0.8475, Val Loss: 1.0299
EarlyStopping counter: 3 out of 20
Epoch 15/1000, Train Loss: 0.8471, Val Loss: 1.0108
EarlyStopping counter: 4 out of 20
Epoch 16/1000, Train Loss: 0.8641, Val Loss: 1.0203
EarlyStopping counter: 5 out of 20
Epoch 17/1000, Train Loss: 0.9979, Val Loss: 1.0186
EarlyStopping counter: 6 out of 20
Epoch 18/1000, Train Loss: 0.9010, Val Loss: 1.0372
EarlyStopping counter: 7 out of 20
Epoch 19/1000,

/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([32])) that is different to the input size (torch.Size([32, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([1, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 4/1000, Train Loss: 1.0347, Val Loss: 1.0670
EarlyStopping counter: 2 out of 20
Epoch 5/1000, Train Loss: 1.3468, Val Loss: 1.0106
EarlyStopping counter: 3 out of 20
Epoch 6/1000, Train Loss: 1.2803, Val Loss: 1.0299
EarlyStopping counter: 4 out of 20
Epoch 7/1000, Train Loss: 1.0379, Val Loss: 1.0209
EarlyStopping counter: 5 out of 20
Epoch 8/1000, Train Loss: 1.2133, Val Loss: 1.0440
EarlyStopping counter: 6 out of 20
Epoch 9/1000, Train Loss: 1.0443, Val Loss: 1.0226
EarlyStopping counter: 7 out of 20
Epoch 10/1000, Train Loss: 1.0306, Val Loss: 1.0303
EarlyStopping counter: 8 out of 20
Epoch 11/1000, Train Loss: 1.1031, Val Loss: 1.0362
EarlyStopping counter: 9 out of 20
Epoch 12/1000, Train Loss: 1.0228, Val Loss: 1.0300
EarlyStopping counter: 10 out of 20
Epoch 13/1000, Train Loss: 0.9992, Val Loss: 1.0243
EarlyStopping counter: 11 out of 20
Epoch 14/1000, Train Loss: 1.0509, Val Loss: 1.0338
EarlyStopping counter: 12 out of 20
Epoch 15/1000, Train Loss: 1.1300, Val Loss: 1

In [21]:
models_list

[{'index': 0,
  'model_type': 'KNeighborsRegressor',
  'hyperparams': {},
  'rmse': 0.30109799755193034},
 {'index': 1,
  'model_type': 'KNeighborsRegressor',
  'hyperparams': {},
  'rmse': 0.4657266303774585},
 {'index': 2,
  'model_type': 'KNeighborsRegressor',
  'hyperparams': {},
  'rmse': 0.4090489627021739},
 {'index': 3,
  'model_type': 'KNeighborsRegressor',
  'hyperparams': {},
  'rmse': 0.33176904431885146},
 {'index': 4,
  'model_type': 'Lasso',
  'hyperparams': {'alpha': 0.001},
  'rmse': 0.26280552998679335},
 {'index': 5,
  'model_type': 'Lasso',
  'hyperparams': {'alpha': 0.002},
  'rmse': 0.25729733201674054},
 {'index': 6,
  'model_type': 'Lasso',
  'hyperparams': {'alpha': 0.005},
  'rmse': 0.29671325918853586},
 {'index': 7,
  'model_type': 'Lasso',
  'hyperparams': {'alpha': 0.01},
  'rmse': 0.26053927565601315},
 {'index': 8,
  'model_type': 'Lasso',
  'hyperparams': {'alpha': 0.02},
  'rmse': 0.28471073997471036},
 {'index': 9,
  'model_type': 'Lasso',
  'hyperpar

In [22]:
models_df = pd.DataFrame(models_list)
models_df = models_df.sort_values('rmse', ascending=True).reset_index(drop=True)
final_models = models_df.head(10)
final_models

,index,model_type,hyperparams,rmse
0,46,GaussianProcessRegressor,"{'kernel': Matern(length_scale=1, nu=1.5)}",0.200529
1,47,GaussianProcessRegressor,"{'kernel': Matern(length_scale=1, nu=1.5)}",0.219741
2,37,RandomForestRegressor,"{'max_depth': 32, 'n_estimators': 1000}",0.232535
3,39,XGBRegressor,"{'max_depth': 5, 'n_estimators': 100, 'learnin...",0.232864
4,49,GaussianProcessRegressor,{'kernel': RBF(length_scale=1)},0.237197
5,48,GaussianProcessRegressor,{'kernel': RBF(length_scale=1)},0.249047
6,5,Lasso,{'alpha': 0.002},0.257297
7,16,Ridge,{'alpha': 7},0.258391
8,7,Lasso,{'alpha': 0.01},0.260539
9,19,Ridge,{'alpha': 20},0.262255


In [23]:
def softmax(series):
    exp_x = np.exp(series - series.max())
    return exp_x / exp_x.sum()

In [24]:
final_models['weights'] = softmax(-final_models['rmse'])
final_models

/tmp/ipykernel_387435/2114309755.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_models['weights'] = softmax(-final_models['rmse'])


,index,model_type,hyperparams,rmse,weights
0,46,GaussianProcessRegressor,"{'kernel': Matern(length_scale=1, nu=1.5)}",0.200529,0.104115
1,47,GaussianProcessRegressor,"{'kernel': Matern(length_scale=1, nu=1.5)}",0.219741,0.102134
2,37,RandomForestRegressor,"{'max_depth': 32, 'n_estimators': 1000}",0.232535,0.100835
3,39,XGBRegressor,"{'max_depth': 5, 'n_estimators': 100, 'learnin...",0.232864,0.100802
4,49,GaussianProcessRegressor,{'kernel': RBF(length_scale=1)},0.237197,0.100366
5,48,GaussianProcessRegressor,{'kernel': RBF(length_scale=1)},0.249047,0.099184
6,5,Lasso,{'alpha': 0.002},0.257297,0.098369
7,16,Ridge,{'alpha': 7},0.258391,0.098262
8,7,Lasso,{'alpha': 0.01},0.260539,0.098051
9,19,Ridge,{'alpha': 20},0.262255,0.097883


In [25]:
rmse = sum(final_models['rmse']*final_models['weights'])
rmse

0.2406661136800385